#  Attendance Sheet Generator
Generates one `.docx` per enumerator with:
- Landscape A4
- Header (logo + org + name/period) repeating on **every page**
- Attendance table: S/N · Date · Purpose (6 checkboxes) · Data · Working Place · Remarks
- Footer (Authorized by + 5 signature boxes) repeating on **every page**
- Summary page (same header/footer)
- Excel QC workbook (Matrix · Summary · Legend)

Run cells in order. Only edit the Configuration cell.

## Imports

In [1]:
import os
from datetime import timedelta
from io import BytesIO

import lxml.etree as etree
import pandas as pd

from docx import Document
from docx.shared import Pt, Emu
from docx.enum.section import WD_ORIENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print("✓ Imports OK")


✓ Imports OK


## Configuration — edit here

In [2]:
XLSX_PATH    = r"E:\Travelling Bill form\177_transforming_vulnerability_into_resiliance_clean_29 Jan 2026.dta"
LOGO_PATH    = r"E:\Travelling Bill form\image1.jpeg"  # blank = no logo
OUT_DIR      = r".\attendance_output"

ORG_LINE1    = "BRAC Institute of Governance and Development"
ORG_LINE2    = "BRAC University"
PROJECT_NAME = ""        # small italic note (blank = skip)

WEEKEND_DAYS_RAW   = "friday"          # e.g. "friday" or "friday,saturday"
HOLIDAY_DATES_RAW  = ""                # e.g. "2025-12-04:2025-12-06, 2025-12-01"
BASEWORK_DATES_RAW = ""

# ── Pre-survey days ───────────────────────────────────────────────────────
# How many calendar days to include BEFORE each enumerator's first survey date.
# These rows appear with blank Purpose and no Data marker (manual fill later).
N_PRE_SURVEY = 5

# ── Column mapping overrides ──────────────────────────────────────────────
# Set to None to let the auto-detector find the column; or set to the exact
# column name string if auto-detection fails.
COL_OVERRIDE = {
    "Enumerator ID"   : None,   # e.g. "enu_code"
    "Enumerator Name" : None,   # e.g. "enu_name"
    "Start Date"      : None,   # e.g. "startdate" or "survey_day"
    "End Date"        : None,   # e.g. "enddate"   — set to "" to disable end-date logic
    "Upazila"         : None,   # only used if a column named "upazila" exists
}

DEMO_MODE = False   # True = first enumerator alphabetically only

print("✓ Config saved.")


✓ Config saved.


## Helpers

In [3]:
def parse_dates(raw):
    out = []
    if not raw or not raw.strip(): return out
    for part in raw.split(","):
        part = part.strip()
        if not part: continue
        if ":" in part:
            try:
                a, b = [pd.to_datetime(s.strip()).date() for s in part.split(":",1)]
                d = a
                while d <= b: out.append(d); d += timedelta(days=1)
            except: print(f"  ⚠  Cannot parse range '{part}' — skipped.")
        else:
            try: out.append(pd.to_datetime(part).date())
            except: print(f"  ⚠  Cannot parse date '{part}' — skipped.")
    return out

def parse_weekend_days(raw):
    mapping = {
        "monday":0,"mon":0,"0":0,"tuesday":1,"tue":1,"1":1,
        "wednesday":2,"wed":2,"2":2,"thursday":3,"thu":3,"3":3,
        "friday":4,"fri":4,"4":4,"saturday":5,"sat":5,"5":5,
        "sunday":6,"sun":6,"6":6,
    }
    days = set()
    for p in raw.lower().split(","):
        p = p.strip()
        if p in mapping: days.add(mapping[p])
        else: print(f"  ⚠  Unknown day '{p}' — skipped.")
    return days or {4}

def find_col(df, candidates, label, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns: return c
        if c.lower() in lower_map: return lower_map[c.lower()]
        hits = [orig for lo,orig in lower_map.items() if c.lower() in lo]
        if len(hits)==1: return hits[0]
    if not required: return None
    raise ValueError(
        f"Cannot find column for '{label}'.\n"
        f"Available: {list(df.columns)}\n"
        f"Set COL_OVERRIDE['{label}'] to the correct name."
    )

print("✓ Helpers ready.")


✓ Helpers ready.


## Load dataset

In [4]:
if not os.path.exists(XLSX_PATH):
    raise FileNotFoundError(f"Not found: {XLSX_PATH}")

# ── File ingestion — supports .dta (Stata), .xlsx/.xls/.xlsm (Excel), .csv ──
ext = os.path.splitext(XLSX_PATH)[1].lower()

if ext == ".dta":
    # --- Stata .dta ---
    # read_stata() auto-converts properly-tagged Stata date formats (%td, %tc, %tC)
    # to Python datetime objects.  Variables with no format tag come in as integers
    # and are handled below in the date-parse step.
    reader = pd.io.stata.StataReader(XLSX_PATH)
    df = reader.read(convert_dates=True, convert_categoricals=True)
    # Flatten CategoricalDtype columns to plain strings so column-detection works
    for col in df.select_dtypes(['category']).columns:
        df[col] = df[col].astype(str).replace({'nan': pd.NA})
    _file_type = "Stata .dta"

elif ext in (".xlsx", ".xls", ".xlsm"):
    df = pd.read_excel(XLSX_PATH)
    _file_type = "Excel"

elif ext == ".csv":
    # Try UTF-8 first; fall back to latin-1 (common in survey exports)
    try:
        df = pd.read_csv(XLSX_PATH, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(XLSX_PATH, encoding="latin-1")
        print("  ⚠  UTF-8 decode failed — re-read with latin-1 encoding.")
    _file_type = "CSV"

else:
    raise ValueError(
        f"Unsupported file extension '{ext}'.\n"
        f"Supported: .dta  .xlsx  .xls  .xlsm  .csv"
    )

print(f"Loaded {len(df):,} rows, {len(df.columns)} columns  [{_file_type}]")

# ── Column resolver ───────────────────────────────────────────────────────────
def resolve(label, candidates, required=True):
    if COL_OVERRIDE.get(label) is not None and COL_OVERRIDE[label] != "":
        c = COL_OVERRIDE[label]
        if c not in df.columns: raise ValueError(f"Override '{c}' not found.")
        return c
    if COL_OVERRIDE.get(label) == "":   # explicitly disabled
        return None
    return find_col(df, candidates, label, required)

col_id      = resolve("Enumerator ID",   ["enu_code","enumerator_id","enum_id","enu_id","EnumID"])
col_name    = resolve("Enumerator Name", ["enu_name","enumerator_name","enum_name","enu_nm","EnumName"])
col_start   = resolve("Start Date",      ["startdate","start_date","survey_day","survey_date","date"])
col_end     = resolve("End Date",        ["enddate","end_date","finishdate","finish_date","endtime","end_time"], required=False)
col_upazila = resolve("Upazila",         ["upazila","upazilla"], required=False)

print(f"  ID → '{col_id}'  Name → '{col_name}'")
print(f"  Start → '{col_start}'  End → '{col_end}'  Upazila → '{col_upazila}'")

# ── Date parsing — robust across all three file types ─────────────────────────
# Stata: already datetime if format was tagged; may be int (Stata day-count) if not.
# Excel: may be datetime, date, or string depending on cell format.
# CSV  : always strings — pd.to_datetime handles most formats.
def _parse_date_col(series):
    """Convert a column to datetime, handling Stata integer day-counts if needed."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return series  # already done (Stata tagged date or Excel datetime)
    if pd.api.types.is_integer_dtype(series) or pd.api.types.is_float_dtype(series):
        # Stata %td stores days since 1960-01-01
        try:
            converted = pd.to_datetime(series, unit="D", origin="1960-01-01", errors="coerce")
            # Sanity check: at least 50% of non-null values should land in 1990-2100
            valid = converted.dropna()
            if len(valid) > 0:
                in_range = ((valid.dt.year >= 1990) & (valid.dt.year <= 2100)).mean()
                if in_range >= 0.5:
                    print(f"    (Stata integer date detected — converted from days-since-1960)")
                    return converted
        except Exception:
            pass
    # Fall back: parse as string
    return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)

df[col_start] = _parse_date_col(df[col_start])
bad = df[col_start].isna().sum()
if bad: print(f"  ⚠  {bad} rows with unparseable start date dropped.")
df = df.dropna(subset=[col_start])

if col_end:
    df[col_end] = _parse_date_col(df[col_end])
    mask_bad = df[col_end].isna() | (df[col_end].dt.date < df[col_start].dt.date)
    if mask_bad.sum():
        print(f"  ⚠  {mask_bad.sum()} rows with missing/invalid end date — using start date for those.")
    df.loc[mask_bad, col_end] = df.loc[mask_bad, col_start]

# ── Enumerator name deduplication ────────────────────────────────────────────
id_names = df.groupby(col_id)[col_name].apply(lambda x: set(x.dropna().unique()))
dups = {k:v for k,v in id_names.items() if len(v)>1}
if dups:
    print("⚠  Duplicate ID with multiple name spellings (using most frequent):")
    for eid,names in dups.items(): print(f"   {eid} → {names}")
    freq = df.groupby(col_id)[col_name].agg(lambda x: x.value_counts().index[0])
    df[col_name] = df[col_id].map(freq)

print(f"Survey span: {df[col_start].min().date()}  →  {df[col_start].max().date()}")
print("✓ Ready.")


Loaded 803 rows, 371 columns  [Stata .dta]
  ID → 'enu_code'  Name → 'enu_name'
  Start → 'startdate'  End → 'enddate'  Upazila → 'None'
Survey span: 2025-11-26  →  2025-12-11
✓ Ready.


## Calendar inputs

In [5]:
weekend_days   = parse_weekend_days(WEEKEND_DAYS_RAW)
holiday_dates  = set(parse_dates(HOLIDAY_DATES_RAW))
basework_dates = set(parse_dates(BASEWORK_DATES_RAW))
day_names = {0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}
print(f"Weekend : {', '.join(day_names[d] for d in sorted(weekend_days))}")
print(f"Holidays: {len(holiday_dates)} | Base work: {len(basework_dates)} | Pre-survey: {N_PRE_SURVEY} days")

def classify(d, is_survey):
    if d.weekday() in weekend_days: return "Weekend"
    if d in holiday_dates:          return "Holiday"
    if d in basework_dates:         return "Base work"
    if is_survey:                   return "Survey"
    return ""

print("✓ Calendar ready.")


Weekend : Fri
Holidays: 0 | Base work: 0 | Pre-survey: 5 days
✓ Calendar ready.


## Build payloads

In [6]:
# Build a working frame with start and end dates per row
work_df = df[[col_id, col_name, col_start] + ([col_end] if col_end else [])].copy()
work_df["_start"] = work_df[col_start].dt.date
work_df["_end"]   = work_df[col_end].dt.date if col_end else work_df[col_start].dt.date

if col_upazila:
    upz = df[[col_id, col_start, col_upazila]].copy()
    upz["_date"] = upz[col_start].dt.date
    upz = upz.dropna(subset=[col_upazila])
    place_map = (upz.groupby([col_id,"_date"])[col_upazila]
                 .agg(lambda x: x.value_counts().index[0] if len(x) else ""))
else:
    place_map = None

# For each enumerator build a set of all 'data dates' (expanding start→end ranges)
def expand_dates(grp):
    """Return set of all dates covered by any [start, end] interval in this group."""
    dates = set()
    for _, r in grp.iterrows():
        d = r["_start"]
        while d <= r["_end"]:
            dates.add(d)
            d += timedelta(days=1)
    return dates

grouped = (
    work_df.groupby([col_id, col_name])
    .apply(expand_dates)
    .reset_index()
    .rename(columns={0: "data_dates"})
)

if DEMO_MODE:
    grouped = grouped.sort_values(col_name).head(1)
    print(f"[DEMO] {grouped.iloc[0][col_name]}")

os.makedirs(OUT_DIR, exist_ok=True)
payloads = []

for _, row in grouped.iterrows():
    enu_id      = str(row[col_id]).strip()
    enu_name    = str(row[col_name]).strip()
    data_dates  = row["data_dates"]          # full set of dates with data

    # Span: from earliest data date to latest data date
    span_start = min(data_dates)
    span_end   = max(data_dates)

    rows, sn = [], 1

    # Pre-survey rows (blank purpose, no data)
    if N_PRE_SURVEY > 0:
        d = span_start - timedelta(days=N_PRE_SURVEY)
        while d < span_start:
            rows.append({"sn":sn, "date":d.strftime("%d %b %Y"),
                         "purpose":"", "has_data":"", "working_place":""})
            sn += 1; d += timedelta(days=1)

    # Survey span rows — every calendar day from span_start to span_end
    d = span_start
    while d <= span_end:
        is_survey = d in data_dates
        wp = ""
        if place_map is not None and is_survey:
            try: wp = str(place_map.loc[(row[col_id], d)])
            except: wp = ""
        rows.append({
            "sn": sn,
            "date": d.strftime("%d %b %Y"),
            "purpose": classify(d, is_survey),
            "has_data": "Yes" if is_survey else "",
            "working_place": wp,
        })
        sn += 1; d += timedelta(days=1)

    # 5 blank buffer rows at the end (for manual additions)
    for _ in range(5):
        rows.append({"sn":"","date":"","purpose":"","has_data":"","working_place":""})

    p_start = (span_start - timedelta(days=N_PRE_SURVEY)).strftime("%d %b %Y") \
              if N_PRE_SURVEY > 0 else span_start.strftime("%d %b %Y")

    payloads.append({
        "enu_id": enu_id, "enu_name": enu_name,
        "period_start": p_start, "period_end": span_end.strftime("%d %b %Y"),
        "rows": rows,
    })

print(f"✓ {len(payloads)} payload(s)")
for p in payloads:
    print(f"   {p['enu_name']} ({p['enu_id']})  "
          f"rows={len(p['rows'])-5}  data={sum(1 for r in p['rows'] if r['has_data']=='Yes')}")


✓ 12 payload(s)
   Md Zubaer (3753)  rows=20  data=13
   Iqbal Hossain (4059)  rows=20  data=13
   Md Mokhsed Ali (4148)  rows=21  data=14
   Tutuli Akter (4383)  rows=20  data=14
   Fatema (4621)  rows=20  data=14
   Shahana khatun (4623)  rows=21  data=14
   Md. Mahedi Hasan (4671)  rows=18  data=12
   Parvin sultana (4713)  rows=20  data=14
   Sharmin Khatun (4723)  rows=21  data=14
   Sharmin Akter (5071)  rows=20  data=14
   Jakia Akter (5115)  rows=21  data=13
   Md. Al Mamun (5180)  rows=18  data=12


C:\Users\user\AppData\Local\Temp\ipykernel_25880\708895942.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(expand_dates)


## XML helpers

In [7]:
# ── Page constants ─────────────────────────────────────────────────────────
DXA          = 635          # 1 DXA = 635 EMU
PAGE_W_DXA   = 15840        # Landscape A4
PAGE_H_DXA   = 12240
MARGIN_DXA   = 720
CONTENT_DXA  = PAGE_W_DXA - 2*MARGIN_DXA  # 14400
COL_SN,COL_DATE,COL_PURP,COL_DATA,COL_WORK,COL_REM = 500,1500,6600,600,2400,2800
assert COL_SN+COL_DATE+COL_PURP+COL_DATA+COL_WORK+COL_REM == CONTENT_DXA, \
    f"Cols sum {COL_SN+COL_DATE+COL_PURP+COL_DATA+COL_WORK+COL_REM} != {CONTENT_DXA}"

def dxa_emu(d): return d * DXA

# ── Table / cell sizing ────────────────────────────────────────────────────
def _upsert_tblW(tbl_el, dxa):
    tblPr = tbl_el.find(qn("w:tblPr"))
    if tblPr is None: tblPr = OxmlElement("w:tblPr"); tbl_el.insert(0,tblPr)
    tblW = tblPr.find(qn("w:tblW"))
    if tblW is None: tblW = OxmlElement("w:tblW"); tblPr.insert(0,tblW)
    tblW.set(qn("w:w"),str(dxa)); tblW.set(qn("w:type"),"dxa")

def set_tbl_grid(tbl_el, col_widths):
    """Set tblGrid so Word renders columns at their actual widths, not equal splits."""
    # Remove existing tblGrid
    for old in tbl_el.findall(qn("w:tblGrid")): tbl_el.remove(old)
    grid = OxmlElement("w:tblGrid")
    for w in col_widths:
        gc = OxmlElement("w:gridCol"); gc.set(qn("w:w"), str(w)); grid.append(gc)
    # tblGrid must come after tblPr
    tblPr = tbl_el.find(qn("w:tblPr"))
    if tblPr is not None: tblPr.addnext(grid)
    else: tbl_el.insert(0, grid)

def _upsert_tcW(tc_el, dxa):
    tcPr = tc_el.find(qn("w:tcPr"))
    if tcPr is None: tcPr = OxmlElement("w:tcPr"); tc_el.insert(0,tcPr)
    tcW = tcPr.find(qn("w:tcW"))
    if tcW is None: tcW = OxmlElement("w:tcW"); tcPr.insert(0,tcW)
    tcW.set(qn("w:w"),str(dxa)); tcW.set(qn("w:type"),"dxa")

def set_cell_border(cell, bordered=True):
    tcPr = cell._tc.get_or_add_tcPr()
    for old in tcPr.findall(qn("w:tcBorders")): tcPr.remove(old)
    tcB = OxmlElement("w:tcBorders")
    for side in ("top","start","bottom","end"):
        el = OxmlElement(f"w:{side}")
        if bordered:
            el.set(qn("w:val"),"single"); el.set(qn("w:sz"),"4")
            el.set(qn("w:space"),"0"); el.set(qn("w:color"),"000000")
        else:
            el.set(qn("w:val"),"none")
        tcB.append(el)
    tcPr.append(tcB)

def set_cell_valign(cell, val="center"):
    tcPr = cell._tc.get_or_add_tcPr()
    for old in tcPr.findall(qn("w:vAlign")): tcPr.remove(old)
    v = OxmlElement("w:vAlign"); v.set(qn("w:val"),val); tcPr.append(v)

def set_cell_margins(cell, top=80, bottom=80, left=120, right=120):
    tcPr = cell._tc.get_or_add_tcPr()
    for old in tcPr.findall(qn("w:tcMar")): tcPr.remove(old)
    m = OxmlElement("w:tcMar")
    for s,v in (("top",top),("start",left),("bottom",bottom),("end",right)):
        el = OxmlElement(f"w:{s}"); el.set(qn("w:w"),str(v)); el.set(qn("w:type"),"dxa"); m.append(el)
    vAlign = tcPr.find(qn("w:vAlign"))
    if vAlign is not None: vAlign.addprevious(m)
    else: tcPr.append(m)

def set_no_table_borders(tbl_el):
    tblPr = tbl_el.find(qn("w:tblPr"))
    if tblPr is None: return
    for old in tblPr.findall(qn("w:tblBorders")): tblPr.remove(old)
    tblB = OxmlElement("w:tblBorders")
    for s in ("top","start","bottom","end","insideH","insideV"):
        el = OxmlElement(f"w:{s}"); el.set(qn("w:val"),"none"); tblB.append(el)
    tblLook = tblPr.find(qn("w:tblLook"))
    if tblLook is not None: tblLook.addprevious(tblB)
    else: tblPr.append(tblB)

def set_row_height(row, dxa):
    trPr = row._tr.get_or_add_trPr()
    for old in trPr.findall(qn("w:trHeight")): trPr.remove(old)
    trH = OxmlElement("w:trHeight")
    trH.set(qn("w:val"),str(dxa)); trH.set(qn("w:hRule"),"atLeast"); trPr.append(trH)


def set_repeat_header(row):
    """Mark a table row as a repeating header so Word repeats it on every page."""
    trPr = row._tr.get_or_add_trPr()
    if trPr.find(qn("w:tblHeader")) is None:
        th = OxmlElement("w:tblHeader")
        th.set(qn("w:val"), "1")
        trPr.append(th)

# ── Paragraph formatting ───────────────────────────────────────────────────
def para_align(para, alignment="center"):
    pPr = para._p.get_or_add_pPr()
    for old in pPr.findall(qn("w:jc")): pPr.remove(old)
    jc = OxmlElement("w:jc"); jc.set(qn("w:val"),alignment); pPr.append(jc)

def para_spacing(para, before=0, after=0, exact_line=True):
    """Set paragraph spacing. exact_line=True overrides docDefault line spacing bloat."""
    pPr = para._p.get_or_add_pPr()
    for old in pPr.findall(qn("w:spacing")): pPr.remove(old)
    spc = OxmlElement("w:spacing")
    spc.set(qn("w:before"), str(before))
    spc.set(qn("w:after"),  str(after))
    if exact_line:
        spc.set(qn("w:line"),     "240")   # single line (no bloat)
        spc.set(qn("w:lineRule"), "exact")
    jc = pPr.find(qn("w:jc"))
    if jc is not None: jc.addprevious(spc)
    else: pPr.append(spc)

def zero_para_spacing(para):
    """Zero before/after + exact single line — used on all table cell paragraphs."""
    para_spacing(para, before=0, after=0, exact_line=True)

def set_tab_stop(para, tab_type, pos_dxa):
    pPr = para._p.get_or_add_pPr()
    for old in pPr.findall(qn("w:tabs")): pPr.remove(old)
    tabs = OxmlElement("w:tabs"); tab = OxmlElement("w:tab")
    tab.set(qn("w:val"),tab_type); tab.set(qn("w:pos"),str(pos_dxa)); tabs.append(tab)
    pPr.insert(0,tabs)

def add_tab(para):
    r = OxmlElement("w:r"); t = OxmlElement("w:tab"); r.append(t); para._p.append(r)

def add_run(para, text, bold=False, size_pt=10, italic=False):
    run = para.add_run(text)
    run.bold=bold; run.italic=italic; run.font.size=Pt(size_pt); run.font.name="Arial"
    return run

# ── Cell configurators ─────────────────────────────────────────────────────
def configure_cell(cell, width_dxa, text="", bold=False, size_pt=10,
                   center=False, top=80, bottom=80, left=120, right=120, bordered=True):
    _upsert_tcW(cell._tc, width_dxa)
    set_cell_border(cell, bordered)
    set_cell_margins(cell, top=top, bottom=bottom, left=left, right=right)
    set_cell_valign(cell)
    p = cell.paragraphs[0]
    zero_para_spacing(p)
    if center: para_align(p, "center")
    if text != "" or text == 0:
        add_run(p, str(text), bold=bold, size_pt=size_pt)

def purpose_cell(cell, width_dxa):
    _upsert_tcW(cell._tc, width_dxa)
    set_cell_border(cell); set_cell_margins(cell,top=80,bottom=80,left=120,right=60)
    set_cell_valign(cell)
    p = cell.paragraphs[0]; zero_para_spacing(p)
    B = "\u2610"
    txt = f"{B} Training   {B} Survey   {B} Base work   {B} Travel   {B} Office work"
    run = p.add_run(txt); run.font.size=Pt(9); run.font.name="Arial"

def data_cell(cell, has_data, width_dxa):
    _upsert_tcW(cell._tc, width_dxa)
    set_cell_border(cell); set_cell_margins(cell,top=80,bottom=80,left=40,right=40)
    set_cell_valign(cell)
    p = cell.paragraphs[0]; zero_para_spacing(p); para_align(p,"center")
    if has_data=="Yes":
        run=p.add_run("Yes"); run.bold=True; run.font.size=Pt(9); run.font.name="Arial"

# ── Settings zoom fix ──────────────────────────────────────────────────────
def fix_zoom(doc):
    for zoom in doc.settings.element.findall(qn("w:zoom")):
        if not zoom.get(qn("w:percent")): zoom.set(qn("w:percent"),"100")

print("✓ XML helpers ready.")


✓ XML helpers ready.


## Logo helper

In [8]:
def register_logo(section):
    """Register logo image directly with the header part — creates header1.xml.rels so
    the image resolves on ALL pages including page 1. Returns rId or None."""
    if not LOGO_PATH or not os.path.exists(LOGO_PATH): return None
    with open(LOGO_PATH,"rb") as f: img=f.read()
    rId, _ = section.header.part.get_or_add_image(BytesIO(img))
    return rId

def logo_anchor_xml(rId):
    W,H=914400,484632
    return (
        f'<w:r xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main"><w:drawing>'
        f'<wp:anchor xmlns:wp="http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing"'
        f' behindDoc="0" distT="0" distB="0" distL="114935" distR="114935"'
        f' simplePos="0" locked="0" layoutInCell="1" allowOverlap="1" relativeHeight="2">'
        f'<wp:simplePos x="0" y="0"/>'
        f'<wp:positionH relativeFrom="column"><wp:posOffset>-95250</wp:posOffset></wp:positionH>'
        f'<wp:positionV relativeFrom="paragraph"><wp:posOffset>-123825</wp:posOffset></wp:positionV>'
        f'<wp:extent cx="{W}" cy="{H}"/><wp:effectExtent l="0" t="0" r="0" b="0"/><wp:wrapNone/>'
        f'<wp:docPr id="1" name="Logo"/>'
        f'<wp:cNvGraphicFramePr>'
        f'<a:graphicFrameLocks xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" noChangeAspect="1"/>'
        f'</wp:cNvGraphicFramePr>'
        f'<a:graphic xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main">'
        f'<a:graphicData uri="http://schemas.openxmlformats.org/drawingml/2006/picture">'
        f'<pic:pic xmlns:pic="http://schemas.openxmlformats.org/drawingml/2006/picture">'
        f'<pic:nvPicPr><pic:cNvPr id="1" name="Logo"/>'
        f'<pic:cNvPicPr><a:picLocks noChangeAspect="1"/></pic:cNvPicPr></pic:nvPicPr>'
        f'<pic:blipFill>'
        f'<a:blip xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" r:embed="{rId}"/>'
        f'<a:stretch><a:fillRect/></a:stretch></pic:blipFill>'
        f'<pic:spPr><a:xfrm><a:off x="0" y="0"/><a:ext cx="{W}" cy="{H}"/></a:xfrm>'
        f'<a:prstGeom prst="rect"><a:avLst/></a:prstGeom></pic:spPr>'
        f'</pic:pic></a:graphicData></a:graphic></wp:anchor></w:drawing></w:r>'
    )

print("✓ Logo helpers ready.")


✓ Logo helpers ready.


## Header & footer builders

In [9]:
def build_header(section, logo_rId, enu_name, enu_id, period_start, period_end):
    """Populate Word Header — repeats automatically on every page."""
    section.different_first_page_header_footer = False
    hdr = section.header
    for p in list(hdr.paragraphs): p._element.getparent().remove(p._element)
    mid = CONTENT_DXA // 2

    def nhp():
        p_el = OxmlElement("w:p"); hdr._element.append(p_el)
        from docx.text.paragraph import Paragraph as P
        return P(p_el, hdr._element)

    # Org line 1 + logo
    p1 = nhp(); set_tab_stop(p1,"center",mid); zero_para_spacing(p1)
    if logo_rId: p1._p.append(etree.fromstring(logo_anchor_xml(logo_rId)))
    add_tab(p1); add_run(p1, ORG_LINE1, bold=True, size_pt=14)

    # Org line 2
    p2 = nhp(); set_tab_stop(p2,"center",mid); zero_para_spacing(p2)
    add_tab(p2); add_run(p2, ORG_LINE2, bold=True, size_pt=14)

    # Title
    p3 = nhp(); para_align(p3,"center"); para_spacing(p3,before=0,after=60,exact_line=True)
    add_run(p3, "Attendance Sheet", size_pt=14)

    # Name / designation / mobile
    p4 = nhp(); zero_para_spacing(p4)
    add_run(p4, "Name: ",              bold=True, size_pt=11)
    add_run(p4, f"{enu_name} ({enu_id})",          size_pt=11)
    add_run(p4, "    Designation: ",   bold=True, size_pt=11)
    add_run(p4, "Enumerator",                      size_pt=11)
    add_run(p4, "    Mobile No.: ",    bold=True, size_pt=11)

    # Period
    p5 = nhp(); para_spacing(p5, before=0, after=80, exact_line=True)
    add_run(p5, "Period  ", bold=True, size_pt=11)
    add_run(p5, f"{period_start}  to  {period_end}", size_pt=11)


def build_footer(section):
    """Populate Word Footer — repeats automatically on every page."""
    ftr = section.footer
    for p in list(ftr.paragraphs): p._element.getparent().remove(p._element)

    def nfp():
        p_el = OxmlElement("w:p"); ftr._element.append(p_el)
        from docx.text.paragraph import Paragraph as P
        return P(p_el, ftr._element)

    # "Authorized by:" label
    auth = nfp(); para_spacing(auth, before=160, after=60, exact_line=True)
    add_run(auth, "Authorized by:", bold=True, size_pt=10)

    # Signature table (built via temp doc then appended to footer element)
    SIG  = ["Enumerator","FA","RA","DA"]
    n    = len(SIG); w = CONTENT_DXA // n
    wids = [w]*(n-1) + [CONTENT_DXA - w*(n-1)]

    tmp     = Document(); sig_tbl = tmp.add_table(rows=2, cols=n)
    _upsert_tblW(sig_tbl._tbl, CONTENT_DXA)
    set_tbl_grid(sig_tbl._tbl, wids)
    set_no_table_borders(sig_tbl._tbl)

    for i, (name, wdxa) in enumerate(zip(SIG, wids)):
        # Row 0: space + line
        cell0 = sig_tbl.rows[0].cells[i]
        _upsert_tcW(cell0._tc, wdxa)
        set_cell_border(cell0, bordered=False)
        set_cell_margins(cell0, top=0, bottom=0, left=80, right=80)
        para_spacing(cell0.paragraphs[0], before=0, after=160, exact_line=True)
        for _ in range(2):
            np2 = cell0.add_paragraph(); para_spacing(np2, before=0, after=160, exact_line=True)
        sp = cell0.add_paragraph(); zero_para_spacing(sp)
        run = sp.add_run("________________________"); run.font.size=Pt(9); run.font.name="Arial"
        # Row 1: name label
        cell1 = sig_tbl.rows[1].cells[i]
        _upsert_tcW(cell1._tc, wdxa)
        set_cell_border(cell1, bordered=False)
        set_cell_margins(cell1, top=0, bottom=0, left=80, right=80)
        p1n = cell1.paragraphs[0]; zero_para_spacing(p1n)
        run1 = p1n.add_run(name)
        run1.bold=True; run1.font.size=Pt(9); run1.font.name="Arial"

    ftr._element.append(sig_tbl._tbl)

print("✓ Header/footer builders ready.")


✓ Header/footer builders ready.


## Table builders

In [10]:
def build_attendance_table(doc, rows):
    HDR = [("S/N",COL_SN),("Date",COL_DATE),("Purpose",COL_PURP),
           ("Data",COL_DATA),("Working Place",COL_WORK),("Remarks",COL_REM)]
    tbl = doc.add_table(rows=len(rows)+1, cols=6)
    _upsert_tblW(tbl._tbl, CONTENT_DXA)
    set_tbl_grid(tbl._tbl, [COL_SN, COL_DATE, COL_PURP, COL_DATA, COL_WORK, COL_REM])

    hdr_row = tbl.rows[0]; set_row_height(hdr_row, 400)
    set_repeat_header(hdr_row)
    for ci,(txt,w) in enumerate(HDR):
        configure_cell(hdr_row.cells[ci], w, txt, bold=True, center=True, size_pt=10,
                       left=40 if ci==3 else 120, right=40 if ci==3 else 120)

    for ri, r in enumerate(rows):
        c = tbl.rows[ri+1].cells
        configure_cell(c[0], COL_SN,   r["sn"],           center=True, size_pt=9)
        configure_cell(c[1], COL_DATE, r["date"],          center=True, size_pt=9)
        purpose_cell(c[2], COL_PURP)
        data_cell(c[3], r["has_data"], COL_DATA)
        configure_cell(c[4], COL_WORK, r["working_place"], size_pt=9)
        configure_cell(c[5], COL_REM,  "",                 size_pt=9)


def build_summary_table(doc, enu_name, enu_id, survey_days_count=0):
    LCOL = int(CONTENT_DXA*0.38); MCOL = int(CONTENT_DXA*0.38); RCOL = CONTENT_DXA-LCOL-MCOL
    ROWS = [
        ("Name",              enu_name,                                      ""),
        ("ID No.",            enu_id,                                        ""),
        ("Period",            "",                                            ""),
        ("Survey Days",       str(survey_days_count) if survey_days_count else "", str(survey_days_count) if survey_days_count else ""),
        ("Training Days",     "",                                            ""),
        ("Travel Days",       "",                                            ""),
        ("Base Work Days",    "",                                            ""),
        ("Office Work Days",  "",                                            ""),
        ("Holiday/Off Days",  "",                                            ""),
        ("Weekend Days",      "",                                            ""),
        ("Total Working Days","",                                            ""),
        ("Remarks / Notes",   "",                                            ""),
    ]
    tbl = doc.add_table(rows=len(ROWS)+1, cols=3)
    _upsert_tblW(tbl._tbl, CONTENT_DXA)
    set_tbl_grid(tbl._tbl, [LCOL, MCOL, RCOL])

    # Banner (merge first two cols)
    brow = tbl.rows[0]; set_row_height(brow, 440)
    brow.cells[0].merge(brow.cells[1])
    configure_cell(brow.cells[0], LCOL+MCOL, "Enumerator Summary",
                   bold=True, center=True, size_pt=11, top=100, bottom=100)
    configure_cell(brow.cells[2], RCOL, "Days Count",
                   bold=True, center=True, size_pt=9.5, top=100, bottom=100)

    for ri,(label,value,dcount) in enumerate(ROWS):
        lc,vc,dc = tbl.rows[ri+1].cells
        configure_cell(lc, LCOL, label, bold=True, size_pt=9.5)
        configure_cell(vc, MCOL, str(value), size_pt=9.5)
        configure_cell(dc, RCOL, str(dcount), center=True, size_pt=9.5)

print("✓ Table builders ready.")


✓ Table builders ready.


## Generate documents

In [11]:
def generate_doc(payload):
    enu_id       = payload["enu_id"]
    enu_name     = payload["enu_name"]
    period_start = payload["period_start"]
    period_end   = payload["period_end"]
    rows         = payload["rows"]

    doc = Document()
    fix_zoom(doc)

    sec = doc.sections[0]
    sec.orientation   = WD_ORIENT.LANDSCAPE
    sec.page_width    = Emu(dxa_emu(PAGE_W_DXA))
    sec.page_height   = Emu(dxa_emu(PAGE_H_DXA))
    sec.top_margin    = Emu(dxa_emu(MARGIN_DXA))
    sec.bottom_margin = Emu(dxa_emu(MARGIN_DXA))
    sec.left_margin   = Emu(dxa_emu(MARGIN_DXA))
    sec.right_margin  = Emu(dxa_emu(MARGIN_DXA))

    # Register logo through the header part (creates header1.xml.rels — works on ALL pages)
    logo_rId = register_logo(sec)

    # Header + footer repeat on every page automatically
    build_header(sec, logo_rId, enu_name, enu_id, period_start, period_end)
    build_footer(sec)

    # Page 1+: attendance table
    build_attendance_table(doc, rows)

    # Page break → summary page
    doc.add_page_break()
    survey_days_count = sum(1 for r in rows if r["has_data"] == "Yes")
    build_summary_table(doc, enu_name, enu_id, survey_days_count)

    safe = enu_name.replace(" ","_").replace("/","_")
    fname = f"{enu_id}_{safe}_attendance.docx"
    doc.save(os.path.join(OUT_DIR, fname))
    return fname


print("Generating…")
for payload in payloads:
    fname = generate_doc(payload)
    data_yes = sum(1 for r in payload["rows"] if r["has_data"]=="Yes")
    print(f"  ✓ {fname}  (data days: {data_yes})")

print(f"\n✓ Done — {len(payloads)} doc(s) in: {os.path.abspath(OUT_DIR)}")


Generating…


  ✓ 3753_Md_Zubaer_attendance.docx  (data days: 13)
  ✓ 4059_Iqbal_Hossain_attendance.docx  (data days: 13)
  ✓ 4148_Md_Mokhsed_Ali_attendance.docx  (data days: 14)
  ✓ 4383_Tutuli_Akter_attendance.docx  (data days: 14)
  ✓ 4621_Fatema_attendance.docx  (data days: 14)
  ✓ 4623_Shahana_khatun_attendance.docx  (data days: 14)
  ✓ 4671_Md._Mahedi_Hasan_attendance.docx  (data days: 12)
  ✓ 4713_Parvin_sultana_attendance.docx  (data days: 14)
  ✓ 4723_Sharmin_Khatun_attendance.docx  (data days: 14)
  ✓ 5071_Sharmin_Akter_attendance.docx  (data days: 14)
  ✓ 5115_Jakia_Akter_attendance.docx  (data days: 13)
  ✓ 5180_Md._Al_Mamun_attendance.docx  (data days: 12)

✓ Done — 12 doc(s) in: e:\Travelling Bill form\attendance_output


## Excel QC workbook

In [12]:
COLOR_MAP = {
    "Survey":"C6EFCE","Training Day":"FFEB9C","Travel":"E2EFDA",
    "Weekend":"D9D9D9","Base work":"BDD7EE","":"FFFFFF",
}
all_dates = sorted(
    {r["date"] for p in payloads for r in p["rows"] if r["date"]},
    key=lambda s: pd.to_datetime(s)
)

wb    = openpyxl.Workbook()
thin  = Side(style="thin", color="CCCCCC")
bdr   = Border(left=thin,right=thin,top=thin,bottom=thin)
hfill = PatternFill("solid", fgColor="2E4057")
hfont = Font(bold=True, color="FFFFFF", name="Arial", size=10)

# Matrix
ws = wb.active; ws.title="Matrix"
c=ws.cell(1,1,"Enumerator"); c.font=hfont; c.fill=hfill
c.alignment=Alignment(horizontal="center",vertical="center"); c.border=bdr
ws.column_dimensions["A"].width=28; ws.row_dimensions[1].height=36
for ci,d in enumerate(all_dates,start=2):
    c=ws.cell(1,ci,d); c.font=hfont; c.fill=hfill; c.border=bdr
    c.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True)
    ws.column_dimensions[get_column_letter(ci)].width=13
for ri,p in enumerate(payloads,start=2):
    d2p={r["date"]:r["purpose"] for r in p["rows"]}
    nc=ws.cell(ri,1,f"{p['enu_name']} ({p['enu_id']})")
    nc.font=Font(name="Arial",size=10); nc.border=bdr; nc.alignment=Alignment(vertical="center")
    for ci,d in enumerate(all_dates,start=2):
        purpose=d2p.get(d,""); c=ws.cell(ri,ci,purpose)
        c.font=Font(name="Arial",size=9); c.border=bdr
        c.alignment=Alignment(horizontal="center",vertical="center")
        c.fill=PatternFill("solid",fgColor=COLOR_MAP.get(purpose,"FFFFFF"))
ws.freeze_panes="B2"

# Summary
ws2=wb.create_sheet("Summary")
hdrs=["ID","Name","Period Start","Period End","Data Days","Survey Days",
      "Training Days","Travel Days","Weekend Days","Base Work Days","Blank Days"]
for ci,h in enumerate(hdrs,start=1):
    c=ws2.cell(1,ci,h); c.font=hfont; c.fill=hfill
    c.alignment=Alignment(horizontal="center",vertical="center"); c.border=bdr
for ri,p in enumerate(payloads,start=2):
    counts={}
    for r in p["rows"]: counts[r["purpose"]]=counts.get(r["purpose"],0)+1
    data_yes=sum(1 for r in p["rows"] if r["has_data"]=="Yes")
    vals=[p["enu_id"],p["enu_name"],p["period_start"],p["period_end"],
          data_yes,counts.get("Survey",0),counts.get("Training Day",0),
          counts.get("Travel",0),counts.get("Weekend",0),
          counts.get("Base work",0),counts.get("",0)]
    for ci,val in enumerate(vals,start=1):
        c=ws2.cell(ri,ci,val); c.font=Font(name="Arial",size=10); c.border=bdr
        c.alignment=Alignment(vertical="center")
        if ci==5: c.fill=PatternFill("solid",fgColor="C6EFCE")
for ci in range(1,len(hdrs)+1):
    vals=[str(ws2.cell(r,ci).value or "") for r in range(1,len(payloads)+2)]
    ws2.column_dimensions[get_column_letter(ci)].width=min(max(len(v) for v in vals)+4,30)
ws2.freeze_panes="A2"; ws2.row_dimensions[1].height=28

# Legend
ws3=wb.create_sheet("Legend")
ws3.column_dimensions["A"].width=16; ws3.column_dimensions["B"].width=32
ws3.cell(1,1,"Colour").font=Font(bold=True,name="Arial")
ws3.cell(1,2,"Meaning").font=Font(bold=True,name="Arial")
for i,(lbl,color) in enumerate([(k,v) for k,v in COLOR_MAP.items() if k],start=2):
    c=ws3.cell(i,1,lbl); c.fill=PatternFill("solid",fgColor=color); c.font=Font(name="Arial",size=10)
    ws3.cell(i,2,f"Day type: {lbl}").font=Font(name="Arial",size=10)

qc_path=os.path.join(OUT_DIR,"_QC_Summary.xlsx"); wb.save(qc_path)
print(f"✓ QC workbook → {qc_path}")


✓ QC workbook → .\attendance_output\_QC_Summary.xlsx


## Summary preview

In [13]:
preview = []
for p in payloads:
    counts = {}
    for r in p["rows"]: counts[r["purpose"]]=counts.get(r["purpose"],0)+1
    preview.append({
        "ID": p["enu_id"], "Name": p["enu_name"],
        "Period": f"{p['period_start']} → {p['period_end']}",
        "Span": len(p["rows"])-5,
        "Data": sum(1 for r in p["rows"] if r["has_data"]=="Yes"),
        "Survey": counts.get("Survey",0), "Weekend": counts.get("Weekend",0),
        "Base work": counts.get("Base work",0),
        "Blank": counts.get("",0),
    })
pd.DataFrame(preview)


,ID,Name,Period,Span,Data,Survey,Weekend,Base work,Blank
0,3753,Md Zubaer,22 Nov 2025 → 11 Dec 2025,20,13,12,2,0,11
1,4059,Iqbal Hossain,21 Nov 2025 → 10 Dec 2025,20,13,12,2,0,11
2,4148,Md Mokhsed Ali,21 Nov 2025 → 11 Dec 2025,21,14,13,2,0,11
3,4383,Tutuli Akter,22 Nov 2025 → 11 Dec 2025,20,14,12,2,0,11
4,4621,Fatema,22 Nov 2025 → 11 Dec 2025,20,14,12,2,0,11
5,4623,Shahana khatun,21 Nov 2025 → 11 Dec 2025,21,14,13,2,0,11
6,4671,Md. Mahedi Hasan,24 Nov 2025 → 11 Dec 2025,18,12,11,1,0,11
7,4713,Parvin sultana,22 Nov 2025 → 11 Dec 2025,20,14,12,2,0,11
8,4723,Sharmin Khatun,21 Nov 2025 → 11 Dec 2025,21,14,13,2,0,11
9,5071,Sharmin Akter,22 Nov 2025 → 11 Dec 2025,20,14,12,2,0,11
